In [ ]:
import os

In [ ]:
os.listdir()

['.config', 'sample_data']

In [ ]:
import kagglehub

path = kagglehub.dataset_download("thedevastator/comprehensive-medical-q-a-dataset")

print("Path to dataset files:", path)

100%|██████████| 4.89M/4.89M [00:01<00:00, 3.33MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2


In [ ]:
os.listdir()

['.config', 'pdf', 'sample_data']

In [ ]:
DATA_DIR = '/root/.cache/kagglehub/datasets/'

In [ ]:
!pip install -q langchain langchain-community pandas

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, CSVLoader

In [ ]:
for root, dirs, files in os.walk("/root/.cache/kagglehub/datasets/"):
    print(root, "->", files[:5])

/root/.cache/kagglehub/datasets/ -> []
/root/.cache/kagglehub/datasets/thedevastator -> []
/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset -> ['2.complete']
/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions -> []
/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2 -> ['train.csv']


In [ ]:
loader = DirectoryLoader(
    path = DATA_DIR,
    glob="**/*.csv",
    loader_cls = CSVLoader
)

In [ ]:
documents = loader.load()

# Some Random Data


In [ ]:
print(f"Total documents {len(documents)}")
print(type(documents))
print(type(documents[0]))

Total documents 16407
<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [ ]:
documents[0]

Document(metadata={'source': '/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2/train.csv', 'row': 0}, page_content='qtype: susceptibility\nQuestion: Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?\nAnswer: LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.')

In [ ]:
print(f'a document\'s page_content {documents[8].page_content}')
print(f'a document\'s metadata {documents[8].metadata}')
print(f'a page content\'s string lenghth {len(documents[8].page_content)}')

a document's page_content qtype: exams and tests
Question: How to diagnose Parasites - Cysticercosis ?
Answer: If you think that you may have cysticercosis, please see your health care provider. Your health care provider will ask you about your symptoms, where you have travelled, and what kinds of foods you eat. The diagnosis of neurocysticercosis usually requires MRI or CT brain scans. Blood tests may be useful to help diagnose an infection, but they may not always be positive in light infections.
  
If you have been diagnosed with cysticercosis, you and your family members should be tested for intestinal tapeworm infection. See the taeniasis section for more information on intestinal tapeworm infections.  
  
More on: Taeniasis
  
More on: Resources for Health Professionals: Diagnosis
a document's metadata {'source': '/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2/train.csv', 'row': 8}
a page content's string lenghth 771


# Data Chunking

In [ ]:
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=80
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 33284


In [ ]:
print(type(chunks[0]))
print('printing some chunks...')

for chunk in chunks[0:5]:
  print(chunk)
  print("="*300)

<class 'langchain_core.documents.base.Document'>
printing some chunks...
page_content='qtype: susceptibility
Question: Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?
Answer: LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.' metadata={'source': '/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2/train.csv', 'row': 0}
page_content='qtype: symptoms
Question: What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?
Answer: LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection

# Sentence Embeddings and Storing Vector DB

In [ ]:
!pip -q install -U sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
!pip -q install "google-cloud-logging>=3.0.0" "google-cloud-appengine-logging>=1.0.0"
!pip -q install "opentelemetry-exporter-gcp-logging>=1.9.0a0,<2.0.0"


In [ ]:
!apt-get update -y
!apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,361 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,477 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.s

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve > /content/ollama.log 2>&1 &
!sleep 3


In [ ]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[]}

In [ ]:
import os
import torch
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

PERSIST_DIR = "/content/db"
COLLECTION = "healthcare_qa"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device is {device}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={
        "batch_size": 2048,
        "normalize_embeddings": True
    },
)

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION,
)

vectordb.persist()
print("Saved ChromaDB to:", PERSIST_DIR)

Device is cuda


/tmp/ipython-input-2489510034.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saved ChromaDB to: /content/db


/tmp/ipython-input-2489510034.py:31: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


# Installing LLM in Local Store

In [ ]:
!

In [ ]:
!ollama serve

/bin/bash: line 1: ollama: command not found


In [ ]:
!ollama pull mistral-nemo:12b
!ollama run mistral-nemo:12b "what is huggingface and what's the difference between it and langchain"


Hugging Face is a technology company based in New York and Paris, specializing in Natural Language Processing (NLP). They are known for several open-source projects and products that facilitate NLP tasks. Here are some of their notable contributions:

1. **Transformers Library**: This is Hugging Face's flagship library for training and using state-of-the-art natural language models. It provides pre-trained models, tokenizers, and tools to build and train custom models.

2. **Model Hub**: A hub where developers can share, discover, and use pre-trained models for various NLP tasks like text classification, question answering, translation, etc.

3. **Datasets Library**: A fast, easy-to-use, and open-source collection of datasets and evaluation metrics for NLP.

4. **Tokenizers**: Hugging Face provides tokenizers for various languages and models, which are crucial for encoding text data into a format that can be understood by neural networks.

LangChain, on the other hand, is an open-sour

# Reimporting VectorDB & Embedding

In [ ]:
import torch
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PERSIST_DIR = "/content/db"
COLLECTION = "healthcare_qa"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load same embedding model
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={
        "batch_size": 512 if device == "cuda" else 64,
        "normalize_embeddings": True
    },
)

# Load vector database
vectordb = Chroma(
    collection_name=COLLECTION,
    persist_directory=PERSIST_DIR,
    embedding_function=embeddings,
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Similarity Searching

In [ ]:
query = "What are common symptoms of diabetes?"

docs_scores = vectordb.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(docs_scores, 1):
    print(f"Result Number: {i} | Similarity Score:{score:.4f}")
    print(doc.page_content[:500])


Result Number: 1 | Similarity Score:0.3813
qtype: symptoms
Question: What are the symptoms of Diabetes ?
Result Number: 2 | Similarity Score:0.4949
Answer: Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild that you dont notice them. An estimated 8 million people in the United States have type 2 diabetes and dont know it, according to 2012 estimates by the Centers for Disease Control and Prevention (CDC). Common Signs Some common symptoms of diabetes are: - being very thirsty  - frequent urination  - feeling very hungry or tired  - losing weight 
Result Number: 3 | Similarity Score:0.4986
qtype: symptoms
Question: What are the symptoms of Diabetes ?
Answer: Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent need to urinate and/or fatigue. Some lose weight without trying. Additional signs include sores that heal slowly, dry, itchy skin, l

# RAG Part

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


retriever = vectordb.as_retriever(search_kwargs={"k": 5})

llm = ChatOllama(model="mistral-nemo:12b", temperature=0)


prompt = ChatPromptTemplate.from_template(
    """You are a healthcare Q&A assistant.
Use ONLY the provided context to answer. If the answer is not in the context, say "I don't know from the provided documents."

Return:
1) A concise answer
2) Bullet list of citations as [1], [2], ... referencing the sources

Context:
{context}

Question:
{question}
"""
)

def format_context(docs):
    blocks = []
    for idx, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown_source")
        row = d.metadata.get("row", "")
        blocks.append(f"[{idx}] (source={src}, row={row})\n{d.page_content}")
    return "\n\n".join(blocks)

# RUNNING
question = "What are the common symptoms of diabetes?"
docs = retriever.invoke(question)

context = format_context(docs)
response = (prompt | llm | StrOutputParser()).invoke({"context": context, "question": question})

print(response)

print("Retrieved sources")
for i, d in enumerate(docs, 1):
    print(f"[{i}] source={d.metadata.get('source')} row={d.metadata.get('row')}")


**Answer:** The common symptoms of diabetes include:

- Being very thirsty
- Frequent urination
- Feeling very hungry or tired
- Losing weight without trying
- Having sores that heal slowly
- Having dry, itchy skin
- Loss of feeling or tingling in the feet
- Having blurry eyesight

**Citations:**
[1], [3]
Retrieved sources
[1] source=/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2/train.csv row=3700
[2] source=/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2/train.csv row=3700
[3] source=/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2/train.csv row=3706
[4] source=/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2/train.csv row=13833
[5] source=/root/.cache/kagglehub/datasets/thedevastator/comprehensive-medical-q-a-dataset/versions/2/train.csv row=15639
